# Stage 3.1 — the baseline run

The reference every later run is compared against (B10). The suite is
executed once at full `n` (seeds at 8, follow-ups at 4, D21). Every execution
is checked structurally and scored point-wise, and the author adopts **one
execution per case** as that case's baseline reply, in `cases/baseline.json`.

This notebook shows, per case, what the run did (structural outcomes, scores,
cost, latency), shows a failure rather than hiding it (§3.5), puts the
top-scoring reply in front of the author for review, and checks that the
adoption is complete.

**The parameters decide which run it describes.** Pointed at a test baseline
(any file outside `cases/`), it says so in every output that matters.

In [1]:
import sys, pathlib, textwrap
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import pandas as pd

from src import survey, comparison as cmp
from src.cases import load_cases
from src.scoring import first_reply, _turn_counts
from src import db

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

# --- parameters -------------------------------------------------------------
# The runs the baseline is drawn from. The real baseline is ONE run at full n.
# The values below are a pipeline rehearsal: 42cab17a (29 cases, n = 1) and the
# D82 probe (the two multi-step seeds, n = 3), which together cover all 31.
BASELINE_RUN_IDS = ["42cab17a-4903-4e39-b00c-ace76b7929e4",
                    "287e7780-2409-450e-a5d1-234b43146390"]
# cases/baseline.json is the author's adoption; anything else is a test file.
BASELINE_FILE = cmp.RUNS_DIR / "baseline_test.json"
# Write the proposal to BASELINE_FILE. Refused for the adopted file: the author
# adopts by instruction, not by a notebook cell.
WRITE_PROPOSAL = True
# ---------------------------------------------------------------------------

TEST = cmp.is_test_baseline(BASELINE_FILE)
print("TEST BASELINE - a rehearsal, not the adopted baseline" if TEST else "ADOPTED BASELINE (cases/baseline.json)")
for rid in BASELINE_RUN_IDS:
    print()
    print(survey.headline(rid))
cases = {c.case_id: c for c in load_cases()}
print()
print(f"suite: {len(cases)} cases")

TEST BASELINE - a rehearsal, not the adopted baseline



run 42cab17a  started 2026-09-23T17:52:07
  notes         FULL SUITE n=1 post-D74..D80 - verification
  agent         gemini-3.5-flash (released 2026-05-19)
  api           0.1.3 @ 5af96abf452b
  suite         031589fd9658
  executions    29  (+0 superseded)
  failed        0
  scored        29, mean 9.34
  cost          $0.6498



run 287e7780  started 2026-09-24T07:27:36
  notes         PROBE multi-step seeds n=3 (D82) - NOT a baseline; unreviewed cases
  agent         gemini-3.5-flash (released 2026-05-19)
  api           0.1.3 @ 5af96abf452b
  suite         71d11f9749ba
  executions    6  (+0 superseded)
  failed        0
  scored        6, mean 8.17
  cost          $0.3467



suite: 31 cases


## 1. The run records its configuration

Every version field must be set, or the comparison in stage 3.2 cannot show
that nothing changed between the runs. Where the baseline spans several runs
(only in a rehearsal), fields that differ between them are listed, because a
real baseline cannot have them.

In [2]:
VERSION_FIELDS = ["model_version", "model_released_on", "prompt_version", "tool_version",
                  "suite_version", "api_version", "api_commit", "app_version", "git_sha",
                  "simulator_model", "simulator_prompt_version", "judge_model", "judge_prompt_version"]
runs = pd.DataFrame([db.admin().table("study_runs").select("*").eq("id", r).execute().data[0]
                     for r in BASELINE_RUN_IDS]).set_index("id")
versions = runs[VERSION_FIELDS].T
versions.columns = [c[:8] for c in versions.columns]
missing = [f for f in VERSION_FIELDS if runs[f].isna().any()]
differ = [f for f in VERSION_FIELDS if runs[f].nunique(dropna=False) > 1]
print(f"version fields unset: {missing or 'none'}")
print(f"fields that differ between the baseline's runs: {differ or 'none'}")
versions

version fields unset: none
fields that differ between the baseline's runs: ['suite_version']


,42cab17a,287e7780
model_version,gemini-3.5-flash,gemini-3.5-flash
model_released_on,2026-05-19,2026-05-19
prompt_version,3ca333b6763e,3ca333b6763e
tool_version,2e6f598f22a1,2e6f598f22a1
suite_version,031589fd96588100ebf8103e1a91b8bb0d9a275849cef8e74b59ab9bdede7fb5,71d11f9749ba7afff2ba363384f78969729fba769e8dbc94bc26657e927ecdce
api_version,0.1.3,0.1.3
api_commit,5af96abf452b,5af96abf452b
app_version,0.1.1,0.1.1
git_sha,5af96abf452bc8994361e13623216a7ed7eb0fec,5af96abf452bc8994361e13623216a7ed7eb0fec
simulator_model,gemini-3.5-flash,gemini-3.5-flash


## 2. What each case did

One row per case: structural outcomes across its repetitions, the point-wise
scores, cost and latency.

**`later turn acted (not judged)`**: how many of the case's executions did
real work *after* the first turn, i.e. a later turn called a tool (drafted a
plan, moved a stop). The judge scores the first reply only (D34, D85), so
for these executions the score describes the opening reply and **not** that
later work. The structural checks do cover it. A later turn that only says
"no thanks" is not counted, because nothing was left unjudged. Non-zero only
for the two multi-step seeds, whose second turn is the point of the case.

In [3]:
frame = pd.concat([survey.run_frame(r).assign(run=r[:8]) for r in BASELINE_RUN_IDS], ignore_index=True)
executions = cmp._executions(BASELINE_RUN_IDS)
by_exec = {e["id"]: e for e in executions}
acted_later = {e["case_id"]: 0 for e in executions}
for e in executions:
    acted_later[e["case_id"]] += _turn_counts(e)["later_acting_turns"] > 0

per_case = frame.groupby("case").agg(
    function=("function", "first"), provenance=("provenance", "first"),
    n=("repetition", "count"),
    structural_none=("structural", lambda s: int((s == "none").sum())),
    structural_detected=("structural", lambda s: int((s == "detected").sum())),
    failed=("blame", lambda s: int((s != "-").sum())),
    score_mean=("score", "mean"), score_min=("score", "min"), score_max=("score", "max"),
    cost_usd=("cost_usd", "sum"), latency_s=("latency_ms", lambda s: round(s.mean() / 1000, 1)),
)
per_case["later turn acted (not judged)"] = per_case.index.map(acted_later)
not_run = sorted(set(cases) - set(per_case.index))
print(f"{len(frame)} executions over {len(per_case)} cases; cases never executed: {not_run or 'none'}")
per_case.round(2)

35 executions over 31 cases; cases never executed: none


,function,provenance,n,structural_none,structural_detected,failed,score_mean,score_min,score_max,cost_usd,latency_s,later turn acted (not judged)
case,,,,,,,,,,,,
advisory_packing_list,advisory,seeded,1,1,0,0,10.00,10,10,0.04,24.7,0
advisory_packing_list__inv1_paraphrase,advisory,metamorphic,1,1,0,0,10.00,10,10,0.03,124.9,0
advisory_packing_list__inv3_irrelevant_detail,advisory,metamorphic,1,1,0,0,7.00,7,7,0.04,62.7,0
advisory_sintra_full_or_half_day,advisory,seeded,1,1,0,0,10.00,10,10,0.02,47.5,0
advisory_sintra_full_or_half_day__inv1_paraphrase,advisory,metamorphic,1,1,0,0,10.00,10,10,0.02,32.1,0
advisory_sintra_full_or_half_day__inv4_german,advisory,metamorphic,1,1,0,0,10.00,10,10,0.03,137.8,0
factual_qa_alfama_history,factual_qa,seeded,1,1,0,0,10.00,10,10,0.02,36.4,0
factual_qa_alfama_history__dir1_the_earthquake,factual_qa,metamorphic,1,1,0,0,10.00,10,10,0.02,11.7,0
factual_qa_alfama_history__inv1_paraphrase,factual_qa,metamorphic,1,1,0,0,10.00,10,10,0.02,24.1,0


## 3. A failure, shown

§3.5: one worked example of a failure is worth more than three of successes.
A structurally failed execution where there is one, else the lowest-scoring.

A structural failure is in what the agent **did**, so that is what is shown,
in the order the checker reads it: what the case asserts, then turn by turn
what the member asked, which tools ran, what the draft held and what changed
on the live itinerary, then the checker's findings. The reply comes last,
because the judge's view of this execution is not where it failed.

In [4]:
from src.execution import load
from src.checker import _slot_entries

def show_execution(ex_row, case):
    e = load(ex_row)
    checks = case.declared_constraints.checks.model_dump(exclude_none=True)
    print("THE CASE ASSERTS")
    for key in ("expected_tool_sequence",):
        print(f"  {key}: {getattr(case, key)}")
    for key, value in checks.items():
        print(f"  {key}: {value}")
    ids = {v: k for k, v in e.blocks.items()}
    for turn in e.turns:
        t = turn["turn_index"]
        print()
        print(f"TURN {t}  member: {(turn.get('user_turn') or '')[:150]!r}")
        for call in (c for c in e.trace if c["turn_index"] == t):
            err = f"  ERROR {str(call['result_error'])[:80]}" if call.get("result_error") else ""
            print(f"  tool  {call['tool']}{err}")
        for draft, dt in zip(e.drafts, e.draft_turns):
            if dt != t:
                continue
            print("  draft proposed:")
            for s in _slot_entries(draft):
                b = s.get("block") or {}
                ref = b.get("reuse_block_id") or ""
                saved = next((title for bid, title in ids.items() if ref and bid.startswith(ref)), None)
                tag = f"   <- reuses saved stop '{saved}'" if saved else ""
                print(f"    {s['_day']} {s.get('start_time')}-{s.get('end_time')}  {b.get('title')}{tag}")
    live = [m for m in e.mutations if m["table_name"] == "itinerary_slots"]
    print()
    print("LIVE ITINERARY CHANGES")
    for m in live or [None]:
        if m is None:
            print("  none")
            break
        pr = m.get("projection") or {}
        print(f"  {m['op']} {ids.get(pr.get('trip_block_id'), pr.get('trip_block_id'))}: "
              f"{pr.get('day_date')} {str(pr.get('start_time'))[:5]}-{str(pr.get('end_time'))[:5]}")
    print()
    print(f"CHECKER: {ex_row.get('structural_outcome')}")
    for f in ex_row.get("structural_findings") or []:
        print("  finding:", f)
    request, reply = first_reply(ex_row)
    print()
    print("FIRST REPLY (what the judge scored)")
    print(textwrap.fill(reply[:600], 110))

bad = frame[frame["structural"] == "detected"]
pick = (bad if len(bad) else frame.dropna(subset=["score"])).sort_values("score").head(1)
if pick.empty:
    print("no execution to show")
else:
    row = pick.iloc[0]
    ex = next(e for e in executions if e["case_id"] == row["case"] and e["repetition"] == row["repetition"]
              and e["run_id"][:8] == row["run"])
    print(f"{row['case']} rep {row['repetition']} (run {row['run']}), judge score {row['score']}")
    print()
    show_execution(ex, cases[row["case"]])

scheduling_afternoon_then_tram rep 2 (run 287e7780), judge score 9



THE CASE ASSERTS
  expected_tool_sequence: ['search_places', 'propose_plan', 'move_slot']
  tool_counts: {'search_places': {'min': 1, 'max': 4}, 'propose_plan': {'min': 1, 'max': 2}, 'move_slot': {'min': 1, 'max': 2}, 'get_weather': {'min': 0, 'max': 1}}
  tool_errors: corrected
  turn_errors: none
  places_surfaced: {'min': 1}
  places_near: {'anchor': 'Tram 28 ride through Alfama', 'within_km': 3.0}
  slot_days: ['2026-11-06']
  slot_count: {'min': 2, 'max': 3}
  draft_slot_starts_between: ('14:00', '18:00')
  slot_moved: {'block': 'Tram 28 ride through Alfama', 'day': '2026-11-06', 'starts_between': ('15:00', '19:30'), 'duration_minutes': {'min': 45, 'max': 90}, 'clear_of_other_slots': True, 'after_latest_draft': True}
  edits: {'move': {'min': 1, 'max': 2}, 'unschedule': 0, 'schedule': 0}
  mutation_counts: {'trip_blocks.INSERT': 0, 'itinerary_slots.INSERT': 0, 'itinerary_slots.UPDATE': {'min': 1, 'max': 2}, 'ai_suggestions.INSERT': {'min': 1, 'max': 2}}
  mutation_changed_keys: {'

**Reading it: a subtle failure, and why it matters that it is caught automatically.**

Nothing in this execution is wrong on its face:

- The tram was moved exactly as asked: Friday, one hour, after the plan (the
  live change above).
- The lunch in the turn-2 draft is **real**. It is the trip's own saved stop,
  already scheduled on Friday, at its correct time, 12:30–14:00.
- Read as a picture of the day, the new draft is even *more* accurate than
  the first one: lunch, then the three Alfama stops, and it hangs together.
- The judge scored the first reply 9/10, and a reader skimming the reply or the
  draft would see nothing amiss.

The problem only shows when you ask what the draft **does**. A draft is not a
description of the day; it is a set of proposals the group accepts slot by
slot. Every slot in it is an offer to *add* something. The lunch slot points
at the saved lunch (`reuses saved stop`), and accepting it schedules that stop
a second time (`draft-accept.ts` passes a reused stop straight to
`addBlockToDay`). Friday would then show the lunch twice at 12:30, and the
trip would carry a conflict nobody created on purpose.

**The rule the agent broke:** a stop that is already scheduled stays where it
is unless the member clearly asks to change it. Here she asked to move the
tram and nothing else. The lunch was never mentioned, so it should not have
entered the draft at all, however accurate its time. (The same run's first
repetition also put the tram itself into the re-draft, right after moving it
live, which would duplicate the tram the same way.)

**Why the suite can see it when a person or the judge would not:** the
checker reads the draft's *arguments*, not its prose. The member asked for
"two or three stops" in the afternoon, so the case asserts a draft of two to
three slots, all starting between 14:00 and 18:00. A draft that smuggles in
an already-scheduled stop breaks both at once: 4 slots, one at 12:30. That is
the structural signal. It is indirect, since the findings name a count and a
time window rather than the duplication, but it fires every time this
behaviour occurs. It fired in 2 of 3 probe repetitions and again in the n = 1
run.

## 4. The proposed baseline, for review

Per case, the execution `comparison.propose_baseline` would adopt: structurally
clean first, then the highest judge score, then the earliest repetition. Its
first reply is printed, because that is the text later runs are judged against.
**The author reads these and adopts, or picks another execution.** Adoption is
the author's act (B10).

In [5]:
proposal = cmp.propose_baseline(BASELINE_RUN_IDS)
unclean = proposal[proposal["structural"] != "none"]
print(f"proposal covers {len(proposal)} of {len(cases)} cases; "
      f"cases whose best execution is not structurally clean: {list(unclean.index) or 'none'}")
proposal[["run_id", "repetition", "structural", "score"]].assign(run_id=lambda f: f["run_id"].str[:8])

proposal covers 31 of 31 cases; cases whose best execution is not structurally clean: none


,run_id,repetition,structural,score
case,,,,
advisory_packing_list,42cab17a,1,none,10
advisory_packing_list__inv1_paraphrase,42cab17a,1,none,10
advisory_packing_list__inv3_irrelevant_detail,42cab17a,1,none,7
advisory_sintra_full_or_half_day,42cab17a,1,none,10
advisory_sintra_full_or_half_day__inv1_paraphrase,42cab17a,1,none,10
advisory_sintra_full_or_half_day__inv4_german,42cab17a,1,none,10
factual_qa_alfama_history,42cab17a,1,none,10
factual_qa_alfama_history__dir1_the_earthquake,42cab17a,1,none,10
factual_qa_alfama_history__inv1_paraphrase,42cab17a,1,none,10


In [6]:
for case_id, row in proposal.iterrows():
    request, reply = first_reply(by_exec[row["execution_id"]])
    print("=" * 110)
    print(f"{case_id}   [score {row['score']}, rep {row['repetition']}, run {row['run_id'][:8]}]")
    print("request:", request)
    print(textwrap.fill(reply[:700] + (" ..." if len(reply) > 700 else ""), 110))

advisory_packing_list   [score 10, rep 1, run 42cab17a]
request: @Tonic what should the three of us pack for this trip?
For your trip in early November (Nov 6–9), Lisbon and Sintra will be transitioning into late autumn. You can
expect mild daytime temperatures (highs around 18°C / 64°F) but cooler evenings (lows around 11°C / 53°F).
November is also historically Lisbon's wettest month, averaging about 134 mm of rain, and Sintra's
microclimate in the hills is notoriously cooler, windier, and mistier.   Here is what the three of us should
pack:  *   **Footwear is priority #1:** Lisbon's famous *calçada portuguesa* (cobblestone streets) are
incredibly slippery, especially when wet, and Sintra involves steep walking. Pack comfortable, broken-in
walking shoes or sneakers with excellent grip. Leave any thin heels  ...
advisory_packing_list__inv1_paraphrase   [score 10, rep 1, run 42cab17a]
request: @Tonic we're trying to work out what to bring — any thoughts for the three of us?
For your tr

advisory_packing_list__inv3_irrelevant_detail   [score 7, rep 1, run 42cab17a]
request: @Tonic what should the three of us pack for this trip? we just bought a huge new suitcase so space isn't really a problem
Since you have plenty of space in that new suitcase, you can pack comfortably for November in Lisbon and
Sintra! November is the wettest month of the year in Lisbon (averaging around 134 mm of rain), with mild
daytime highs of 18°C (64°F) and cooler nights around 12°C (54°F). Sintra is up in the hills and will feel
noticeably cooler, windier, and damper.   Here is what the three of us should pack:  *   **Footwear with
serious grip:** Lisbon's classic *calçada portuguesa* (cobblestone streets) are notoriously slippery,
especially when wet. Leave the heels at home and pack sturdy, worn-in sneakers or boots with excellent
traction for walking up and down the steep hills. *   **Rain gear & layer ...
advisory_sintra_full_or_half_day   [score 10, rep 1, run 42cab17a]
request: @Tonic is

advisory_sintra_full_or_half_day__inv1_paraphrase   [score 10, rep 1, run 42cab17a]
request: @Tonic do you reckon Sintra needs a whole day, or would half of one do?
Sintra definitely deserves a **full day**.   Because it is located up in the hills and has several major,
spread-out landmarks (like Pena Palace, the Moorish Castle, and Quinta da Regaleira), trying to squeeze it
into a half-day is incredibly rushed. Between the 40-minute train ride each way from Lisbon, navigating the
local tourist buses, and actually walking the steep, expansive palace grounds, a half-day would barely leave
you time to see one site and grab lunch.   Since you already have the magnificent **Pena Palace** saved and
scheduled for 10:00 AM on Day 3 (Sunday, Nov 8), I highly recommend keeping that day dedicated to Sintra so
you can explore the historic town center or add anothe ...


advisory_sintra_full_or_half_day__inv4_german   [score 10, rep 1, run 42cab17a]
request: @Tonic lohnt sich Sintra für einen ganzen Tag oder reicht ein halber?
Sintra lohnt sich definitiv für einen **ganzen Tag**! Ein halber Tag reicht kaum aus, um dem Zauber dieses
UNESCO-Weltkulturerbes gerecht zu werden.   Die Region ist berühmt für ihre weitläufigen, romantischen
Paläste, Burgen und mystischen Gärten. Da die Wege zwischen den Sehenswürdigkeiten in den Bergen steil sind
und man oft mit Wartezeiten oder Shuttlebussen rechnen muss, verliert man an einem halben Tag zu viel Zeit mit
der Logistik.   Allein für den farbenfrohen **Pena-Palast** (den ihr bereits für Tag 3 auf dem Plan habt) und
seine riesigen Parkanlagen solltet ihr mindestens 3 Stunden einplanen. Wenn ihr einen ganzen Tag bleibt, könnt
ihr das perfekt mit einem zweiten Highlight verbin ...


factual_qa_alfama_history   [score 10, rep 1, run 42cab17a]
request: @Tonic what's the history of Alfama, briefly?
Alfama is Lisbon's oldest neighborhood, and during Moorish rule, it actually made up the entire city before it
expanded westward. Because of its solid rock foundation, it miraculously survived the devastating 1755 Lisbon
earthquake, leaving its medieval labyrinth of narrow streets, alleys, and small squares intact. Over the
centuries, it evolved into a traditional district for fishermen and the working class, and today it is famous
as the historic heart of Fado music.
factual_qa_alfama_history__dir1_the_earthquake   [score 10, rep 1, run 42cab17a]
request: @Tonic what happened to Alfama in the 1755 earthquake?
Unlike most of Lisbon, Alfama actually survived the devastating 1755 earthquake largely intact! Because it was
built on a solid foundation of compact bedrock, the neighborhood was spared from the worst of the shaking and
the subsequent tsunami, which is why it still 

factual_qa_alfama_history__inv1_paraphrase   [score 10, rep 1, run 42cab17a]
request: @Tonic give me the short version of how Alfama came to be what it is
Alfama's name comes from the Arabic *al-ḥamma*, meaning "hot springs," dating back to Moorish rule when the
district made up the entire city of Lisbon. Because its solid hill foundation survived the devastating 1755
earthquake, it remains a historic, labyrinthine maze of narrow streets that transitioned over the centuries
into a traditional home for fishermen and the working class.
factual_qa_lisbon_forecast_next_days   [score 10, rep 1, run 42cab17a]
request: @Tonic what's the weather doing in Lisbon over the next couple of days?
It is looking very warm and mostly sunny in Lisbon over the next couple of days!   Today (Wednesday) and
tomorrow (Thursday) will be mainly clear with highs reaching around 33°C (91°F) and lows overnight dropping to
about 19–21°C. On Friday, it will turn overcast with temperatures cooling down slightly to a

... (9 outputs left out)

venue_lookup_seafood_lisbon__dir1_and_a_hotel   [score 10, rep 1, run 42cab17a]
request: /find good seafood places near Cais do Sodré, nothing too pricey — and somewhere to stay nearby too
Here are a few great seafood spots and budget-friendly places to stay right around Cais do Sodré:  **Seafood
Spots (Casual & Great Value)** *   **Sol e Pesca**: A quirky, casual favorite set in a former fishing tackle
shop on the Pink Street, famous for its incredible gourmet canned Portuguese seafood and cold beers. *   **Uma
Marisqueira**: Just a short walk up towards Baixa, this is a legendary, no-frills spot famous for serving just
one main thing: a massive, incredibly delicious pot of wet seafood rice (*arroz de marisco*). *
**Marisqueira Azul - Time Out Market**: Located right inside the market, offering fresh oysters, clams, and
garlic shrimp at the counter without the high-end sit- ...


venue_lookup_seafood_lisbon__dir4_not_a_meal   [score 10, rep 1, run 42cab17a]
request: /find where can we get a drink by the river near Cais do Sodre?
Here are three great spots right by the river or just steps away near Cais do Sodré:  *   **Quiosque Ribeira
das Naus**: A relaxed outdoor kiosk right on the riverfront promenade with deck chairs, perfect for a sunset
beer or cocktail with direct views of the Tagus. *   **La Terraza**: A highly-rated spot very close to the
waterfront offering great cocktails, tapas, and a lively atmosphere. *   **Pensão Amor**: Just a block back
from the river on Pink Street, this famous, eclectically decorated former brothel is a must-visit for a unique
indoor cocktail experience.  Would you like me to pre-schedule one of these for drinks after your Time Out
Market lunch on Day 1, or perhaps for another eve ...
venue_lookup_seafood_lisbon__inv2_typos   [score 10, rep 1, run 42cab17a]
request: /find god seafod palces near cais do sodre, nothin too price

venue_lookup_sintra_lunch   [score 7, rep 1, run 42cab17a]
request: /find where should we eat lunch on the Sintra day? somewhere near the palace, proper local food
Here are three great, authentic local spots not too far from Pena Palace for your Sintra day trip:  *
**Taxka**: A fantastic, highly-rated tavern serving up incredible local petiscos (Portuguese tapas) in a very
cozy, traditional atmosphere. * **Tasco do Strauss**: A wonderful, friendly spot known for hearty, comforting
Portuguese dishes and great local wines. * **Tasca d'Ana**: A charming, rustic tavern offering classic, home-
style Portuguese cooking that feels like a meal in a local home.  Would you like me to schedule one of these
for lunch on Day 3 (Nov 8) right after your Pena Palace visit?


venue_lookup_sintra_lunch__dir1_coffee_not_lunch   [score 8, rep 1, run 42cab17a]
request: /find where can we get a coffee and a pastry near the palace on the Sintra day?
Here are three great spots near the historic center of Sintra (just a short drive or bus ride down from Pena
Palace) to grab a coffee and a pastry:  *   **Folar da Ruby**: A highly-rated, cozy local gem famous for its
incredible traditional Portuguese cakes, pastries, and warm hospitality. *   **Ca.fé Coffee House**: A
fantastic specialty coffee shop with excellent espresso drinks, delicious pastries, and a relaxed vibe. *
**Pau de Canela Sintra**: A classic Portuguese pastry shop and bakery with a wonderful selection of
traditional sweet treats and coffee.  Would you like me to pre-schedule one of these for the morning of Day 3
(November 8th) right before your Pena Palace visit?
venue_lookup_sintra_lunch__dir2_lisbon_day   [score 8, rep 1, run 42cab17a]
request: /find where should we eat lunch on the last day? near t

## 5. Adoption

A test baseline may be written from the proposal here. The adopted one is
written only on the author's instruction, and this cell refuses to write it.

In [7]:
if WRITE_PROPOSAL and TEST:
    cmp.write_baseline(proposal, BASELINE_FILE, by="agent (pipeline rehearsal)",
                       note="TEST baseline - proposal written by 03_baseline, not an adoption")
    print(f"wrote {BASELINE_FILE.relative_to(cmp.RUNS_DIR.parent)}")
elif WRITE_PROPOSAL:
    print("refused: cases/baseline.json is the author's adoption and is not written from a notebook")

baseline = cmp.load_baseline(BASELINE_FILE) if BASELINE_FILE.exists() else {"cases": {}}
entries = baseline.get("cases", {})
gaps = cmp.baseline_gaps(baseline)
differs = [c for c, e in entries.items() if c in proposal.index and e["execution_id"] != proposal.at[c, "execution_id"]]
print(f"baseline file: {'present' if BASELINE_FILE.exists() else 'ABSENT'}; "
      f"adopted by {baseline.get('adopted_by', '-')}, on {baseline.get('on', '-')}")
print(f"cases without a baseline entry: {gaps or 'none'}")
print(f"entries that differ from the proposal (the author chose another execution): {differs or 'none'}")

wrote runs\baseline_test.json


baseline file: present; adopted by agent (pipeline rehearsal), on 2026-09-24
cases without a baseline entry: none
entries that differ from the proposal (the author chose another execution): none


## 6. Done when (B10)

The checks the spec names, computed. A `False` is a gap to close before this
run can serve as the baseline.

In [8]:
done = {
    "every case has a baseline entry": not gaps,
    "every entry is structurally clean": all(by_exec.get(e["execution_id"], {}).get("structural_outcome") == "none"
                                             for e in entries.values()),
    "every execution was scored": frame["score"].notna().all(),
    "every version field is set": not missing,
    "the baseline is one run": len(BASELINE_RUN_IDS) == 1,
    "the baseline is the adopted file": not TEST,
}
pd.Series(done, name="holds").to_frame()

,holds
every case has a baseline entry,True
every entry is structurally clean,True
every execution was scored,True
every version field is set,True
the baseline is one run,False
the baseline is the adopted file,False
